# 14 — HerBERT na TwitterEmo: raw vs korekta (LanguageTool)

Dwa fine-tuny HerBERT-base: tekst surowy vs po LanguageTool. Progi na val, F1-Macro + recall klas rzadkich na teście.

In [ ]:
import warnings
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from scipy.special import expit
from sklearn.metrics import f1_score, recall_score
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, EarlyStoppingCallback)
from thesis_lib import optimal_thresholds as opt_thr
warnings.filterwarnings("ignore")
EMOTIONS=["radość","smutek","zaufanie","wstręt","strach","gniew","przeczuwanie","zdziwienie"]
RARE=["strach","zaufanie","smutek"]; RANDOM_STATE=42
torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
PROCESSED_DIR=Path("../data/processed"); LT=PROCESSED_DIR/"lt"; RESULTS_DIR=Path("../data/results"); HF_OUT=Path("../data/transformers")
MODEL="allegro/herbert-base-cased"; device="cuda" if torch.cuda.is_available() else "cpu"; tok=AutoTokenizer.from_pretrained(MODEL)
def col(sp): 
    d=pd.read_csv(PROCESSED_DIR/f"twitteremo_{sp}.csv"); d["tekst"]=d["tekst"].fillna("")
    lt=pd.read_csv(LT/f"twitteremo_{sp}.csv")["text_lt"].fillna("").tolist()
    return d["tekst"].tolist(),lt,d[EMOTIONS].values
tw={sp:col(sp) for sp in ["train","val","test"]}
y_val,y_test=tw["val"][2],tw["test"][2]
pos=tw["train"][2].sum(0); neg=len(tw["train"][2])-pos
POS_WEIGHT=torch.tensor(np.clip(neg/np.maximum(pos,1),1.0,10.0),dtype=torch.float32)
print("device",device)

In [ ]:
def make_ds(text,y): 
    d=Dataset.from_dict({"text":text,"labels":y.astype("float32").tolist()})
    return d.map(lambda b: tok(b["text"],truncation=True,max_length=128),batched=True,remove_columns=["text"])
class WT(Trainer):
    def __init__(self,*a,pos_weight=None,**k): super().__init__(*a,**k); self.pw=pos_weight
    def compute_loss(self,model,inputs,return_outputs=False,**kw):
        lab=inputs.pop("labels"); out=model(**inputs)
        loss=F.binary_cross_entropy_with_logits(out.logits.float(),lab.float(),pos_weight=self.pw.to(out.logits.device))
        return (loss,out) if return_outputs else loss

In [3]:
def train_variant(var, epochs=4):
    idx={"raw":0,"lt":1}[var]
    ds_tr=make_ds(tw["train"][idx],tw["train"][2]); ds_va=make_ds(tw["val"][idx],y_val); ds_te=make_ds(tw["test"][idx],y_test)
    model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=len(EMOTIONS),problem_type="multi_label_classification")
    args=TrainingArguments(output_dir=str(HF_OUT/f"corr_{var}"),eval_strategy="epoch",save_strategy="epoch",save_total_limit=1,
        load_best_model_at_end=True,metric_for_best_model="f1_macro",greater_is_better=True,per_device_train_batch_size=8,
        per_device_eval_batch_size=32,gradient_accumulation_steps=2,gradient_checkpointing=True,num_train_epochs=epochs,
        learning_rate=2e-5,warmup_ratio=0.1,weight_decay=0.01,fp16=torch.cuda.is_available(),logging_steps=100,report_to="none",seed=RANDOM_STATE)
    cm=lambda p:{"f1_macro":f1_score(p.label_ids.astype(int),(expit(p.predictions)>=0.5).astype(int),average="macro",zero_division=0)}
    tr=WT(model=model,args=args,train_dataset=ds_tr,eval_dataset=ds_va,data_collator=DataCollatorWithPadding(tok),
          compute_metrics=cm,pos_weight=POS_WEIGHT,callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    tr.train()
    thr=opt_thr(y_val,expit(tr.predict(ds_va).predictions)); pred=(expit(tr.predict(ds_te).predictions)>=thr).astype(int)
    row={"wariant":var,"f1_macro":f1_score(y_test,pred,average="macro",zero_division=0),
         "f1_micro":f1_score(y_test,pred,average="micro",zero_division=0)}
    for c in RARE: row[f"recall_{c}"]=recall_score(y_test[:,EMOTIONS.index(c)],pred[:,EMOTIONS.index(c)],zero_division=0)
    del tr,model; torch.cuda.empty_cache(); print(f"  {var}: F1-Macro={row['f1_macro']:.3f}"); return row
rows=[train_variant(v) for v in ["raw","lt"]]
res=pd.DataFrame(rows); res.to_csv(RESULTS_DIR/"correction_herbert_tw.csv",index=False); display(res.round(3))
d=res[res.wariant=="lt"]["f1_macro"].iloc[0]-res[res.wariant=="raw"]["f1_macro"].iloc[0]
(RESULTS_DIR/"correction_herbert_summary.md").write_text(f"# HerBERT TW: raw vs LT\n\n- Δ(LT-raw) F1-Macro = {d:+.3f}\n- Czy transformer korzysta z korekty (char nie korzystał)?")
print(f"Δ(LT-raw) = {d:+.3f}")

Map:   0%|          | 0/28684 [00:00<?, ? examples/s]

Map:   7%|▋         | 2000/28684 [00:00<00:02, 11496.43 examples/s]

Map:  14%|█▍        | 4000/28684 [00:00<00:02, 11545.66 examples/s]

Map:  21%|██        | 6000/28684 [00:00<00:01, 11607.61 examples/s]

Map:  28%|██▊       | 8000/28684 [00:00<00:01, 11634.27 examples/s]

...

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.encoder.layer.0.output.LayerNorm.beta', 'bert.encoder.layer.0.output.LayerNorm.gamma', 'bert.encoder.layer.1.attention.output.LayerNorm.beta', 'bert.encoder.layer.1.attention.output.LayerNorm.gamma', 'bert.encoder.layer.1.output.LayerNorm.beta', 'bert.encoder.layer.1.output.LayerNorm.gamma', 'bert.encoder.layer.2.attention.output.LayerNorm.beta', 'bert.encoder.layer.2.attention.output.LayerNorm.gamma', 'bert.encoder.layer.2.output.LayerNorm.beta', 'bert.encoder.layer.2.output.LayerNorm.gamma', 'bert.encoder.layer.3.attention.output.LayerNorm.beta', 'bert.encoder.layer.3.attention.output.LayerNorm.gamma', 'bert.encoder.layer.3.output.LayerNorm.beta', 'bert.encoder.layer.3.output.LayerNorm.gamma', 'bert.encoder.layer.4.attention.output.LayerNor

  lt: F1-Macro=0.540


,wariant,f1_macro,f1_micro,recall_strach,recall_zaufanie,recall_smutek
0,raw,0.561,0.645,0.385,0.469,0.493
1,lt,0.540,0.641,0.231,0.516,0.522


Δ(LT-raw) = -0.021
